In [2]:
import numpy as np 
import pandas as pd 
from collections import Counter
data_id3 = [
    {'Age':'<=30', 'Income':'High',   'Student':'No',  'Credit':'Fair',      'BuysComputer':'No'},
    {'Age':'<=30', 'Income':'High',   'Student':'No',  'Credit':'Excellent', 'BuysComputer':'No'},
    {'Age':'31-40','Income':'High',   'Student':'No',  'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'No',  'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Low',    'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Low',    'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'No'},
    {'Age':'31-40','Income':'Low',    'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'<=30', 'Income':'Medium', 'Student':'No',  'Credit':'Fair',      'BuysComputer':'No'},
    {'Age':'<=30', 'Income':'Low',    'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'<=30', 'Income':'Medium', 'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'31-40','Income':'Medium', 'Student':'No',  'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'31-40','Income':'High',   'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'No',  'Credit':'Excellent', 'BuysComputer':'No'},
]

In [6]:
import math
def entropy_whole_dataset(data, attr):
    # calculates the entropy of the passed attribute
    column = data[attr]
    counts = Counter(column)
    total = 0
    for c,v in counts.items():
        total += v
    entropy = 0
    for _, p in counts.items():
        entropy -= (p / total) * math.log2(p / total + 1e-15)
    
    return entropy

In [7]:
df = pd.DataFrame(data_id3)

In [8]:
entropy_whole_dataset(df, "BuysComputer")

0.9402859586706281

In [14]:
def entropy_partial_set(data, attr, set_to, target):
    # calculates the entropy of the passed attribute
    subset = data[data[attr] == set_to]
    column = subset[target]
    counts = Counter(column)
    total = 0
    for c,v in counts.items():
        total += v
    entropy = 0
    for _, s in counts.items():
        entropy -= (s / total) * math.log2(s / total + 1e-15)
    
    return entropy

In [17]:
entropy_partial_set(df, 'Age', '<=30', 'BuysComputer')

0.9709505944546657

In [42]:
def info_gain(data, attr, target):
    n = len(data)
    base_entropy = entropy_whole_dataset(data, target)
    values = data[attr].unique()

    weighted = 0
    for val in values:
        subset = data[data[attr] == val]
        weight = len(subset) / n
        weighted += weight * entropy_partial_set(data, attr, val, target)

    return base_entropy - weighted

In [43]:
info_gain(df, "Age", "BuysComputer")

0.24674981977443877

In [44]:
def best_split(data, features, target):
    gains = {}
    for f in features:
        gains[f] = info_gain(data, f, target)
    return max(gains, key = gains.get) if gains else None

In [45]:
best_split(df, ['Age', 'Income', 'Student', 'Credit'], 'BuysComputer')

'Age'

In [46]:
def build_id3_tree(df, features, target):
    root     = best_split(df, features, target)
    labels = list(df[target])
    if len(set(labels)) == 1:
        return labels[0]
    if not features:
        return Counter(labels).most_common(1)[0][0]

    tree = {
        root : {

        }
    }

    for val in df[root].unique():
        subset = df[df[root] == val]
        remaining = [f for f in features if f != root]
        tree[root][val] = build_id3_tree(
            subset, remaining, target        
        )
    return tree

In [47]:
build_id3_tree(df, ['Age', 'Income', 'Student', 'Credit'], 'BuysComputer')

{'Age': {'<=30': {'Student': {'No': 'No', 'Yes': 'Yes'}},
  '31-40': 'Yes',
  '>40': {'Credit': {'Fair': 'Yes', 'Excellent': 'No'}}}}